pipeline 11 лабороторной работы(визуализация данных) 
данные брались из лабороторной работы 13

In [38]:
from pathlib import Path

In [39]:
# этап загрузки
load_code = """
# -*- coding: utf-8 -*-
'''Слой 1: загрузка сырых данных. Читает файлы как есть, ничего не исправляет.'''
from pathlib import Path

import pandas as pd


def load_trips(raw_dir, pattern="trips_*.csv"):
    '''Читает все месячные выгрузки из raw_dir и склеивает в одну таблицу.'''
    files = sorted(Path(raw_dir).glob(pattern))
    if not files:
        raise FileNotFoundError(f"в {raw_dir} нет файлов по шаблону {pattern}")

    frames = []
    for path in files:
        part = pd.read_csv(path, sep=";", encoding="cp1251")
        part["source_file"] = path.name          # помним, откуда пришла строка
        frames.append(part)
        print(f"    {path.name}: {len(part)} строк")

    return pd.concat(frames, ignore_index=True)


def load_carriers(raw_dir, filename="carriers.csv"):
    '''Читает справочник перевозчиков.'''
    return pd.read_csv(Path(raw_dir) / filename)
"""

Path("src/load.py").write_text(load_code.lstrip(), encoding="utf-8")
print("создан src/load.py")

создан src/load.py


In [40]:
#этап обработки 
prepare_code = """
# -*- coding: utf-8 -*-
'''Слой 2: приведение сырых данных к пригодному для анализа виду.'''
import pandas as pd

CANON_CARRIERS = {
    "транслогистик": "ТрансЛогистик",
    "рускарго": "РусКарго",
    "сибпуть": "СибПуть",
    "волгатранс": "ВолгаТранс",
    "автодорсервис": "АвтоДорСервис",
}
TARIFFS = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}


def to_canon(value, canon_map=CANON_CARRIERS):
    '''Приводит название перевозчика к каноническому виду.'''
    s = str(value).strip().lower()
    for prefix in ('ооо "', "ооо ", 'ао "', "ао "):
        if s.startswith(prefix):
            s = s[len(prefix):]
    s = s.strip('"').strip()
    return canon_map.get(s, value)


def to_number(series):
    '''Превращает текст с десятичной запятой в число.'''
    return pd.to_numeric(
        series.astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def prepare_trips(raw, max_cost_ratio=5.0):
    '''Очищает выгрузку. Возвращает (clean_df, report_dict).'''
    report = {"строк получено": len(raw)}
    df = raw.copy()

    df = df.drop_duplicates(subset=["trip_id"], keep="first")
    report["удалено дублей"] = report["строк получено"] - len(df)

    df["cost_rub"] = to_number(df["cost_rub"])
    df["weight_t"] = to_number(df["weight_t"])
    df["delay_h"] = pd.to_numeric(df["delay_h"], errors="coerce").fillna(0)
    df["date"] = pd.to_datetime(df["date"], format="%d.%m.%Y", errors="coerce")
    report["дат не распознано"] = int(df["date"].isna().sum())

    df["carrier"] = df["carrier"].apply(to_canon)
    df["cargo_type"] = df["cargo_type"].str.strip().str.capitalize()
    report["перевозчиков после нормализации"] = int(df["carrier"].nunique())

    bad_weight = df["weight_t"] <= 0
    report["удалено: нулевой вес"] = int(bad_weight.sum())
    df = df[~bad_weight]

    expected = df["weight_t"] * df["distance_km"] * df["cargo_type"].map(TARIFFS)
    bad_cost = (expected / df["cost_rub"]) > max_cost_ratio
    report["удалено: стоимость не бьётся с тарифом"] = int(bad_cost.sum())
    df = df[~bad_cost]

    df["month"] = df["date"].dt.to_period("M").astype(str)
    df["is_delayed"] = df["delay_h"] > 0
    df["cost_per_km"] = (df["cost_rub"] / df["distance_km"]).round(2)
    
    
    if len(df['carrier'].unique())!=5:
        report['неизветсные перевозчики']=sorted(set(df['carrier'].dropna())-set(CANON_CARRIERS.values()))
        print('Есть неизветсные перевозчики!', report['неизветсные перевозчики'])


    report["строк осталось"] = len(df)
    report['пропуски в весе'] = int(df['weight_t'].isna().sum())
    report["доля потерь, %"] = round((1 - len(df) / report["строк получено"]) * 100, 1)
    return df.reset_index(drop=True), report
"""

Path("src/prepare.py").write_text(prepare_code.lstrip(), encoding="utf-8")

2827

In [ ]:
#этап визуализации
visualize_code="""
# -*- coding: utf-8 -*-
'''Слой 3 построение графиков'''
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import config as cfg


def _save_png(fig, out_dir, filename, dpi=cfg.FIGURE_DPI):
    '''Сохраняет рисунок, закрывает его и возвращает путь.'''
    path = Path(out_dir) / filename
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
    finally:
        plt.close(fig)
    return path


def plot_revenue_by_carrier(df, out_dir=cfg.CHARTS_DIR):
    '''Строит столбчатую диаграмму выручки по перевозчикам.'''
    carrier_sorted = df.groupby("carrier")["cost_rub"].sum().sort_values()
    fig, ax = plt.subplots(figsize=cfg.WIDE_FIGSIZE)
    ax.barh(carrier_sorted.index,carrier_sorted.values/ cfg.MILLION)
    ax.set_title('Выручка по перевозчикам')
    ax.set_xlabel('Выурчка, млн. руб')
    ax.set_ylabel('Перевозчики')
    fig.tight_layout()

    return _save_png(fig, out_dir, 'revenue_by_carrier.png')


def plot_trips_by_month(df, out_dir=cfg.CHARTS_DIR):
    '''Показывает, как менялось число рейсов по месяцам.'''
    trips_month = df.groupby('month')['trip_id'].count()
    fig, ax = plt.subplots(figsize=cfg.WIDE_FIGSIZE)
    ax.plot(trips_month.index,trips_month.values,marker=cfg.LINE_MARKER)
    ax.set_title('Число рейсов по месяцам')
    ax.set_xlabel('Год, месяц')
    ax.set_ylabel('Кол-во рейсов')
    fig.tight_layout()

    return _save_png(fig, out_dir, 'trips_by_month.png')


def plot_distance_and_duration(df, out_dir=cfg.CHARTS_DIR):
    '''Показывает связь расстояния со временем в пути.'''
    fig, ax = plt.subplots(figsize=cfg.WIDE_FIGSIZE)
    ax.scatter(df['duration_h'],df['distance_km'],alpha=cfg.SCATTER_ALPHA)
    ax.set_title('Зависимость расстояния от времени в пути')
    ax.set_xlabel('Время в пути, часы')
    ax.set_ylabel('Расстояние, км')
    fig.tight_layout()

    return _save_png(fig, out_dir, 'distance_and_duration.png')


def plot_cost_by_carrier(df, out_dir=cfg.CHARTS_DIR):
    '''Сравнивает распределение стоимости рейса по перевозчикам.'''
    fig, ax = plt.subplots(figsize=cfg.SEABORN_FIGSIZE)
    order = df.groupby("carrier")["cost_rub"].median().sort_values().index
    sns.boxplot(data=df,x='carrier',y='cost_rub',ax=ax, order=order)

    ax.set_title("Распределение стоимости рейса по перевозчикам")
    ax.set_xlabel("")
    ax.set_ylabel("Стоимость рейса, руб.")
    fig.tight_layout()

    return _save_png(fig, out_dir, 'cost_by_carrier.png')


def plot_cargo_weight(df, out_dir=cfg.CHARTS_DIR):
    '''Строит гистограмму веса груза.'''
    fig,ax= plt.subplots(figsize=cfg.SEABORN_FIGSIZE)
    sns.histplot(data=df,x="weight_t")

    ax.set_title("Распределение веса груза.")
    ax.set_xlabel("Вес, тонн")
    ax.set_ylabel("Кол-во рейсов")

    fig.tight_layout()

    return _save_png(fig, out_dir, 'cargo_weight.png')


def plot_cost_and_distance(df, out_dir=cfg.CHARTS_DIR):
    '''Показывает стоимость рейса по расстоянию и типу груза.'''
    fig,ax= plt.subplots(figsize=cfg.SEABORN_FIGSIZE)
    sns.scatterplot(data=df,x='distance_km',y=df['cost_rub'], hue='cargo_type')

    ax.set_title("Распрделение цены по расстоянию рейса и типу груза.")
    ax.set_xlabel("Расстояние, км")
    ax.set_ylabel("Цена, рубли")

    fig.tight_layout()

    return _save_png(fig, out_dir, 'cost_and_distance.png')


def plot_delay_heatmap(df, out_dir=cfg.CHARTS_DIR):
    '''Строит теплокарту средней задержки по маршрутам и перевозчикам.'''
    pivot_t = df.pivot_table(index="route", columns="carrier",
                           values="delay_h", aggfunc="mean", fill_value=0) 

    fig, ax = plt.subplots(figsize=cfg.HEATMAP_FIGSIZE)
    sns.heatmap(pivot_t, annot=True, fmt=cfg.HEATMAP_FORMAT, cmap=cfg.HEATMAP_CMAP, ax=ax,
                cbar_kws={"label": "Средняя задержка, час."})
    ax.set_title("Средняя задержка, час .")
    ax.set_xlabel("Перевозчик")
    ax.set_ylabel("Маршрут")


    fig.tight_layout()

    return _save_png(fig, out_dir, 'delay_heatmap.png')


def plot_revenue_by_route(df, out_dir=cfg.CHARTS_DIR):
    '''Строит интерактивную диаграмму выручки по маршрутам и сохраняет её в HTML.'''
    route_stats = (df.groupby("route")
                   .agg(revenue=("cost_rub", "sum"), trips=("trip_id", "count"))
                   .reset_index()
                   .sort_values("revenue",ascending=False))

    fig = px.bar(route_stats, x="route", y="revenue" ,title="Выручка по маршрутам(наведите курсор на столбец)",
                hover_data=["trips", "route"], labels={'route':'Маршрут','revenue':'Выручка, руб.','trips':'Кол-во рейсов'}
                )

    path = Path(out_dir) / 'revenue_by_route.html'
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(path, include_plotlyjs=True)
    return path


def plot_cost_by_cargo(df, out_dir=cfg.CHARTS_DIR):
    '''Строит интерактивное распределение стоимости рейса по типам груза.'''
    fig = px.box(df, x="cargo_type", y="cost_rub" ,title="Зависимость стоимости рейса от типа груза(наведите курсор на столбец)",
                 labels={'cargo_type':'Тип груза','cost_rub':'Стоимость рейса, руб.'})

    path = Path(out_dir) / 'cost_by_cargo.html'
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(path, include_plotlyjs=True)
    return path


def _draw_revenue_by_month(df, ax):
    '''Рисует выручку по месяцам на оси дашборда.'''
    monthly_rev=df.groupby("month")["cost_rub"].sum()/cfg.MILLION


    ax.plot(monthly_rev.index,monthly_rev.values,marker=cfg.LINE_MARKER)
    ax.set_title('Динамика выручки')
    ax.set_xlabel('Год, месяц')
    ax.set_ylabel('Выручка, млн. рублей')
    ax.set_ylim(*cfg.REVENUE_YLIM)
    for x, y in zip(monthly_rev.index, monthly_rev.values):
        ax.annotate(f"{y:.1f}", xy=(x, y), xytext=(0, 8), textcoords="offset points",
                    ha="center", fontsize=9)

    ax.annotate(cfg.GROWTH_LABEL,
                xy=cfg.GROWTH_XY, xytext=cfg.GROWTH_XYTEXT,
                arrowprops=dict(arrowstyle="->", color="black"))


def plot_revenue_by_month(df, out_dir=cfg.CHARTS_DIR):
    '''Динамика выручки по месяцам.'''
    fig, ax = plt.subplots(figsize=cfg.WIDE_FIGSIZE)
    _draw_revenue_by_month(df, ax)
    fig.tight_layout()

    return _save_png(fig, out_dir, 'revenue_by_month.png')


def _draw_route_revenue_bar(df, ax):
    '''Рисует выручку по маршрутам на оси дашборда.'''

    route_sorted = df.groupby("route")["cost_rub"].sum().sort_values()
    ax.barh(route_sorted.index,route_sorted.values/ cfg.MILLION)
    ax.set_title('Выручка по маршрутам')
    ax.set_xlabel('Выурчка, млн. руб')
    ax.set_ylabel('Маршруты')


def plot_route_revenue_bar(df, out_dir=cfg.CHARTS_DIR):
    '''Столбчатая диаграмма выручки по маршрутам.'''
    fig, ax = plt.subplots(figsize=cfg.HEATMAP_FIGSIZE)
    _draw_route_revenue_bar(df, ax)
    fig.tight_layout()

    return _save_png(fig, out_dir, 'route_revenue_bar.png')


def _draw_trip_cost_histogram(df, ax):
    '''Рисует гистограмму стоимости рейса на оси дашборда.'''

    ax.hist(df['cost_rub']/cfg.THOUSAND,bins=cfg.COST_BINS)
    ax.set_title("Распредление стоимости ")
    ax.set_xlabel("Стоимость рейса, тыс. руб.")
    ax.set_ylabel("Количество рейсов, шт.")


def plot_trip_cost_histogram(df, out_dir=cfg.CHARTS_DIR):
    '''Распределение стоимости рейса.'''
    fig, ax = plt.subplots(figsize=cfg.WIDE_FIGSIZE)
    _draw_trip_cost_histogram(df, ax)
    fig.tight_layout()

    return _save_png(fig, out_dir, 'trip_cost_histogram.png')


def _draw_otd_by_carrier(df, ax, target):
    '''Рисует долю рейсов без задержки и линию цели по OTD.'''
    kpi=df.groupby('carrier').agg(otd=("delay_h", lambda s: (s == 0).mean() * 100))

    ax.barh(kpi.index, kpi['otd'])
    ax.axvline(target, color=cfg.OTD_COLOR, linestyle=cfg.OTD_LINESTYLE, label=f'цель: {target}%')
    if kpi['otd'].max()<target:
        ax.set_title(f'Никто из перевозчиков не достиг цели OTD - {target}%!')
    else:
        ax.set_title(f'OTD по перевозчикам, цель {target}%')
    ax.set_xlabel("OTD, %")
    ax.set_ylabel("Перевозчик")
    ax.set_xlim(*cfg.OTD_XLIM)


def plot_otd_by_carrier(df, out_dir=cfg.CHARTS_DIR, target=cfg.OTD_TARGET):
    '''Доля рейсов без задержки с линией целевого OTD.'''
    fig, ax = plt.subplots(figsize=cfg.WIDE_FIGSIZE)
    _draw_otd_by_carrier(df, ax, target)
    fig.tight_layout()

    return _save_png(fig, out_dir, 'otd_by_carrier.png')


def plot_dashboard(df, out_dir=cfg.CHARTS_DIR, target=cfg.OTD_TARGET):
    '''Собирает четыре графика дашборда и сохраняет их в PNG.'''
    fig, axes = plt.subplots(2, 2, figsize=cfg.DASHBOARD_FIGSIZE)

    _draw_revenue_by_month(df,axes[0,0])
    _draw_route_revenue_bar(df,axes[0,1])
    _draw_trip_cost_histogram(df,axes[1,0])
    _draw_otd_by_carrier(df,axes[1,1],target)

    fig.suptitle(cfg.DASHBOARD_TITLE, fontsize=cfg.DASHBOARD_TITLE_SIZE)
    fig.tight_layout()

    return _save_png(fig, out_dir, 'dashboard.png', dpi=cfg.DASHBOARD_DPI)

"""
Path("src/visualize.py").write_text(visualize_code.lstrip(), encoding="utf-8")

9413

In [42]:
run_code = """
# # -*- coding: utf-8 -*-
'''Точка входа: запускает весь анализ от сырых данных до отчёта.'''
import json
from os.path import relpath
from datetime import datetime
from pathlib import Path

import config
from src.load import load_trips, load_carriers
from src.prepare import prepare_trips
from src import visualize


def df_to_markdown(df):
    '''Превращает таблицу в markdown без внешних зависимостей.'''
    header = "| " + " | ".join(str(c) for c in df.columns) + " |"
    sep = "|" + "---|" * len(df.columns)
    rows = ["| " + " | ".join(str(v) for v in row) + " |"
            for row in df.itertuples(index=False)]
    return chr(10).join([header, sep] + rows)


def build_report(kpi, quality, by_carrier, figures):
    '''Собирает текстовый отчёт в формате Markdown.'''
    lines = ["# Отчёт по перевозкам", ""]
    lines.append("Сформирован: " + datetime.now().strftime("%d.%m.%Y %H:%M"))

    lines += ["", "## Ключевые показатели", ""]
    for k, v in kpi.items():
        lines.append(f"- **{k}**: {v}")

    lines += ["", "## Качество данных", ""]
    for k, v in quality.items():
        lines.append(f"- {k}: {v}")

    lines += ["", "## По перевозчикам", "", df_to_markdown(by_carrier)]

    lines += ["", "## Графики", ""]
    for path in figures:
        link = Path(relpath(path, config.REPORTS_DIR)).as_posix()
        marker = "!" if path.suffix == ".png" else ""
        lines.append(f"{marker}[{path.stem}]({link})")

    return chr(10).join(lines)


def main():
    started = datetime.now()
    print("=" * 56)
    print("ЗАПУСК PIPELINE", started.strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 56)

    for folder in (config.PROCESSED_DIR, config.REPORTS_DIR, config.FIGURES_DIR):
        Path(folder).mkdir(parents=True, exist_ok=True)

    print("[1/5] загрузка")
    raw = load_trips(config.RAW_DIR, config.TRIPS_PATTERN)
    carriers = load_carriers(config.RAW_DIR)

    print("[2/5] подготовка")
    clean, quality = prepare_trips(raw, max_cost_ratio=config.MAX_COST_RATIO)
    clean_path = Path(config.PROCESSED_DIR) / "trips_clean.csv"
    clean.to_csv(clean_path, index=False, encoding="utf-8")
    print(f"    сохранено: {clean_path} ({len(clean)} строк)")

    print("[3/5] расчёт метрик")
    kpi = {
        "рейсов": len(clean),
        "выручка": int(clean["cost_rub"].sum()),
        "средний чек": round(clean["cost_rub"].mean()),
        "медианный чек": round(clean["cost_rub"].median()),
        "выручка на км": round(float(clean["cost_rub"].sum() / clean["distance_km"].sum()), 1),
        "OTD, %": round(float((~clean["is_delayed"]).mean() * 100), 1),
    }

    by = {}
    for column in ("carrier", "month"):
        table = (clean.groupby(column)
                 .agg(trips=("trip_id", "count"),
                      revenue=("cost_rub", "sum"),
                      avg_ticket=("cost_rub", "mean"),
                      otd=("is_delayed", lambda s: (~s).mean() * 100))
                 .round(1)
                 .sort_values("revenue", ascending=False)
                 .reset_index())
        table.to_csv(Path(config.PROCESSED_DIR) / f"kpi_by_{column}.csv",
                     index=False, encoding="utf-8")
        by[column] = table
    by_carrier = by["carrier"]

    print("[4/5] графики")
    fig1 = visualize.plot_revenue_by_month(clean, config.FIGURES_DIR)
    fig2 = visualize.plot_otd_by_carrier(clean, config.FIGURES_DIR, target=config.OTD_TARGET)
    figures = [fig1, fig2]
    for plot in (visualize.plot_revenue_by_carrier, visualize.plot_trips_by_month,
                 visualize.plot_cost_by_carrier, visualize.plot_cargo_weight,
                 visualize.plot_cost_and_distance, visualize.plot_delay_heatmap,
                 visualize.plot_revenue_by_route, visualize.plot_cost_by_cargo,
                 visualize.plot_route_revenue_bar, visualize.plot_trip_cost_histogram):
        figures.append(plot(clean, config.FIGURES_DIR))
    if "duration_h" in clean.columns:
        figures.append(visualize.plot_distance_and_duration(clean, config.FIGURES_DIR))
    else:
        print("    график расстояния и времени пропущен: нет столбца duration_h")
    figures.append(visualize.plot_dashboard(clean, config.FIGURES_DIR, target=config.OTD_TARGET))
    for path in figures:
        print(f"    {path}")

    print("[5/5] отчёт")
    report_path = Path(config.REPORTS_DIR) / "report.md"
    report_path.write_text(build_report(kpi, quality, by_carrier, figures),
                           encoding="utf-8")

    manifest = {
        "запуск": started.isoformat(timespec="seconds"),
        "длительность, с": round((datetime.now() - started).total_seconds(), 2),
        "параметры": {"MAX_COST_RATIO": config.MAX_COST_RATIO,
                      "OTD_TARGET": config.OTD_TARGET,
                      "TRIPS_PATTERN": config.TRIPS_PATTERN},
        "качество данных": quality,
        "итоговые метрики": kpi,
    }
    manifest_path = Path(config.PROCESSED_DIR) / "run_manifest.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2),
                             encoding="utf-8")

    print(f"    {report_path}")
    print(f"    {manifest_path}")
    print("ГОТОВО")
    return manifest


if __name__ == "__main__":
    main()

"""

Path("run.py").write_text(run_code.lstrip(), encoding="utf-8")
print("создан run.py")

создан run.py


In [43]:
import subprocess
import sys

result = subprocess.run([sys.executable, "run.py"], capture_output=True, text=True,
                        encoding="utf-8")
print(result.stdout)
if result.returncode != 0:
    print("ОШИБКА:")
    print(result.stderr[-2000:])

ЗАПУСК PIPELINE 2026-11-24 23:03:11
[1/5] загрузка
    trips_2025_04.csv: 242 строк
    trips_2025_05.csv: 262 строк
[2/5] подготовка
    сохранено: data\processed\trips_clean.csv (477 строк)
[3/5] расчёт метрик
[4/5] графики
    график расстояния и времени пропущен: нет столбца duration_h
    reports\figures\revenue_by_month.png
    reports\figures\otd_by_carrier.png
    reports\figures\revenue_by_carrier.png
    reports\figures\trips_by_month.png
    reports\figures\cost_by_carrier.png
    reports\figures\cargo_weight.png
    reports\figures\cost_and_distance.png
    reports\figures\delay_heatmap.png
    reports\figures\revenue_by_route.html
    reports\figures\cost_by_cargo.html
    reports\figures\route_revenue_bar.png
    reports\figures\trip_cost_histogram.png
    reports\figures\dashboard.png
[5/5] отчёт
    reports\report.md
    data\processed\run_manifest.json
ГОТОВО



In [44]:
print("АРТЕФАКТЫ:")
for p in sorted(Path("data/processed").glob("*")) + sorted(Path("reports").glob("**/*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size // 1024} КБ)")

print()
print("=" * 56)
print(Path("reports/report.md").read_text(encoding="utf-8")[:900])

АРТЕФАКТЫ:
  data\processed\kpi_by_carrier.csv  (0 КБ)
  data\processed\kpi_by_month.csv  (0 КБ)
  data\processed\run_manifest.json  (0 КБ)
  data\processed\trips_clean.csv  (71 КБ)
  reports\figures\cargo_weight.png  (23 КБ)
  reports\figures\cost_and_distance.png  (91 КБ)
  reports\figures\cost_by_cargo.html  (4744 КБ)
  reports\figures\cost_by_carrier.png  (39 КБ)
  reports\figures\dashboard.png  (132 КБ)
  reports\figures\delay_heatmap.png  (74 КБ)
  reports\figures\otd_by_carrier.png  (27 КБ)
  reports\figures\revenue_by_carrier.png  (24 КБ)
  reports\figures\revenue_by_month.png  (22 КБ)
  reports\figures\revenue_by_route.html  (4715 КБ)
  reports\figures\route_revenue_bar.png  (43 КБ)
  reports\figures\trip_cost_histogram.png  (22 КБ)
  reports\figures\trips_by_month.png  (33 КБ)
  reports\report.md  (1 КБ)

# Отчёт по перевозкам

Сформирован: 24.11.2026 23:03

## Ключевые показатели

- **рейсов**: 477
- **выручка**: 21651154
- **средний чек**: 45390
- **медианный чек**: 40731
-